In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

# S3 destination path
S3_PATH = "s3://databricks-data-landing-s3/databricks/coingecko/top_coins"

# CoinGecko API configuration
API_URL = "https://api.coingecko.com/api/v3/coins/markets"
TOP_N_COINS = 10

In [0]:
# Parameters for top coins by market cap
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": TOP_N_COINS,
    "page": 1,
    "sparkline": False
}

print(f"Fetching top {TOP_N_COINS} coins from CoinGecko API...")
response = requests.get(API_URL, params=params)

if response.status_code == 200:
    coins_data = response.json()
    print(f"✓ Successfully fetched {len(coins_data)} coins")
    print(f"  Coins: {', '.join([coin['symbol'].upper() for coin in coins_data[:5]])}...")
else:
    raise Exception(f"API request failed with status code: {response.status_code}\nError: {response.text}")

In [0]:
# Convert to pandas DataFrame
df_pandas = pd.DataFrame(coins_data)

# Convert to Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Add load timestamp for partitioning and tracking
load_timestamp = datetime.now()
df_spark = df_spark.withColumn("load_timestamp", lit(load_timestamp))
df_spark = df_spark.withColumn("load_date", lit(load_timestamp.strftime("%Y-%m-%d")))
df_spark = df_spark.withColumn("load_hour", lit(load_timestamp.strftime("%H")))

print(f"✓ Transformed data: {df_spark.count()} records")
print(f"  Columns: {len(df_spark.columns)}")
print(f"  Partition values: load_date={load_timestamp.strftime('%Y-%m-%d')}, load_hour={load_timestamp.strftime('%H')}")

# Display sample
display(df_spark.limit(5))

In [0]:
print(f"Writing to {S3_PATH}...")

# Write to S3 as parquet with partitioning by load_date and load_hour
df_spark.write \
    .mode("append") \
    .partitionBy("load_date", "load_hour") \
    .parquet(S3_PATH)

print(f"✓ Successfully loaded data to S3!")
print(f"  Location: {S3_PATH}")
print(f"  Records: {df_spark.count()}")
print(f"  Partitions: load_date={load_timestamp.strftime('%Y-%m-%d')}/load_hour={load_timestamp.strftime('%H')}")